# 03. Preprocesamiento y reducción de dimensionalidad
Fase 3 del proyecto. Se prepara el dataset **Airline Passenger Satisfaction** para un futuro modelado mediante
codificación de variables categóricas, escalado de variables numéricas y reducción de dimensionalidad (PCA y t-SNE).
Las decisiones se basan en los resultados del EDA (`02_eda.ipynb`).

## 1. Carga de datos

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer

sns.set_style('whitegrid')

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
df_train = pd.read_csv(os.path.join(path, "train.csv"))
df_test = pd.read_csv(os.path.join(path, "test.csv"))
df = pd.concat([df_train, df_test], ignore_index=True)

print(df.shape)

(129880, 25)


## 2. Limpieza

Se aplican las mismas decisiones del EDA:
- Eliminación de `Unnamed: 0` (índice de los archivos originales) e `id` (identificador sin valor analítico).
- Imputación de los 393 faltantes de `Arrival Delay in Minutes` con la mediana, dado el sesgo de la variable.

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'id'])
df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())

print(df.shape)
print("Faltantes restantes:", df.isnull().sum().sum())

(129880, 23)
Faltantes restantes: 0


**Nota:** la imputación y el escalado se ajustan aquí sobre todo el dataset con fines exploratorios. En la fase de modelado deben ajustarse
solo con el conjunto de entrenamiento para evitar fuga de datos.

## 3. Variable objetivo

`satisfaction` se codifica como 1 (satisfecho) y 0 (neutral o insatisfecho).

In [ ]:
y = (df['satisfaction'] == 'satisfied').astype(int)
X = df.drop(columns=['satisfaction'])

print(y.value_counts())
print("\nProporción de satisfechos:", round(y.mean(), 3))

satisfaction
0    73452
1    56428
Name: count, dtype: int64

Proporción de satisfechos: 0.434


El 43% de los pasajeros está satisfecho; no se considera un desbalance que requiera tratamiento.

## 4. Grupos de variables

In [ ]:
nominales = ['Gender', 'Customer Type', 'Type of Travel']
ordinales = ['Class']
delays = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
continuas = ['Age', 'Flight Distance']
calificaciones = [c for c in X.columns if c not in nominales + ordinales + delays + continuas]

print("Calificaciones (0-5):", len(calificaciones))
for col in nominales + ordinales:
    print(f"{col}: {X[col].unique().tolist()}")

Calificaciones (0-5): 14
Gender: ['Male', 'Female']
Customer Type: ['Loyal Customer', 'disloyal Customer']
Type of Travel: ['Personal Travel', 'Business travel']
Class: ['Eco Plus', 'Business', 'Eco']


| Grupo | Variables | Tratamiento |
|---|---|---|
| Nominales (binarias) | `Gender`, `Customer Type`, `Type of Travel` | One-Hot |
| Ordinal | `Class` (Eco < Eco Plus < Business) | Codificación ordinal + escalado |
| Delays | `Departure Delay`, `Arrival Delay` (skewness ≈ 6.8) | `log1p` + escalado |
| Continuas | `Age`, `Flight Distance` | Escalado |
| Calificaciones | 14 preguntas de encuesta (0 a 5) | Escalado |

## 5. Codificación de variables categóricas

### Variables nominales: One-Hot

Las tres variables son binarias, por lo que se usa `drop='if_binary'` para obtener una sola columna 0/1 por variable y evitar columnas redundantes.

In [ ]:
onehot = OneHotEncoder(drop='if_binary', sparse_output=False)
nominales_cod = pd.DataFrame(onehot.fit_transform(X[nominales]),
                             columns=onehot.get_feature_names_out(nominales))

pd.concat([X[nominales].head(), nominales_cod.head()], axis=1)

,Gender,Customer Type,Type of Travel,Gender_Male,Customer Type_disloyal Customer,Type of Travel_Personal Travel
0,Male,Loyal Customer,Personal Travel,1.0,0.0,1.0
1,Male,disloyal Customer,Business travel,1.0,1.0,0.0
2,Female,Loyal Customer,Business travel,0.0,0.0,0.0
3,Female,Loyal Customer,Business travel,0.0,0.0,0.0
4,Male,Loyal Customer,Business travel,1.0,0.0,0.0


### Variable ordinal: `Class`

`Class` se codifica de forma ordinal (Eco = 0, Eco Plus = 1, Business = 2) para conservar la jerarquía entre clases.
El orden se define explícitamente, ya que por defecto sklearn lo asigna alfabéticamente.

In [ ]:
ordinal = OrdinalEncoder(categories=[['Eco', 'Eco Plus', 'Business']])
clase_cod = ordinal.fit_transform(X[ordinales])

pd.DataFrame({'Class': X['Class'].head(8), 'Class codificada': clase_cod[:8, 0]})

,Class,Class codificada
0,Eco Plus,1.0
1,Business,2.0
2,Business,2.0
3,Business,2.0
4,Business,2.0
5,Eco,0.0
6,Eco,0.0
7,Business,2.0


### Variables de calificación

Las 14 calificaciones son numéricas y ordinales, por lo que se conservan como numéricas. Aplicar One-Hot generaría 84 columnas y eliminaría el orden de la escala.